# Quantum Neural Network Architectures

Compare hardware-efficient and strongly-entangling ansatze, and quantify their entangling power with the Meyer-Wallach measure computed directly from the state vector.

**Objectives:**
- Build a bare product-rotation circuit, a hardware-efficient ansatz, and a strongly-entangling ansatz from Braket primitives.
- Implement the Meyer-Wallach entanglement measure Q from a pure state vector via single-qubit reduced density matrices.
- Verify Q = 0 for product states and Q = 1 for Bell/GHZ states.
- Sample seeded random parameters and compare the average entanglement each ansatz produces.
- Connect average Q to the ideas of entangling capability and expressibility.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first.
# Browser-runnable against qcsim (a pure-NumPy Braket stand-in) -- no AWS needed.

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

# Local simulator (free, instant) for the one illustrative sampling demo.
device = LocalSimulator()

# Global problem size used throughout this notebook.
N_QUBITS = 4
N_LAYERS = 2

np.set_printoptions(precision=4, suppress=True)
print("Setup complete:", N_QUBITS, "qubits,", N_LAYERS, "layers")

## 1. Measuring entanglement: the Meyer-Wallach Q

An ansatz is only as useful as the *correlations* it can create. A circuit that produces only product
states -- where each qubit is independent -- is just a stack of single-qubit rotations and can never
model a joint distribution over qubits. We want a single number that says how entangled, on average,
the states an ansatz produces are.

The **Meyer-Wallach measure** does exactly this. For a pure global state `|psi>` on `n` qubits,

$$
Q = 2 - \frac{2}{n} \sum_{k=0}^{n-1} \mathrm{Tr}\big(\rho_k^2\big),
$$

where `rho_k` is the **single-qubit reduced density matrix** of qubit `k` (trace out every other
qubit). The quantity `Tr(rho_k^2)` is the *purity* of qubit `k`:

- If qubit `k` is unentangled from the rest, `rho_k` is a pure 1-qubit state and its purity is `1`.
- If qubit `k` is maximally entangled with the rest, `rho_k = I/2` and its purity is `1/2`.

So `Q = 0` for a fully product state (all purities `1`) and `Q = 1` when every qubit is maximally
entangled with the others (all purities `1/2`). We compute `rho_k` straight from the state vector by
reshaping it into an `n`-dimensional tensor.

In [ ]:
def state_vector_of(circuit):
    """Return the exact state vector of a circuit as a complex NumPy array.

    qcsim is big-endian: qubit 0 is the leftmost (most-significant) axis.
    """
    return np.asarray(circuit.state_vector(), dtype=complex)


def single_qubit_purity(psi_tensor, k, n):
    """Purity Tr(rho_k^2) of qubit k from an n-axis state tensor (big-endian)."""
    # Move qubit k to the front, flatten the rest: rows index qubit k's two amplitudes.
    axes = [k] + [a for a in range(n) if a != k]
    m = np.transpose(psi_tensor, axes).reshape(2, -1)
    rho_k = m @ m.conj().T            # 2x2 reduced density matrix (partial trace)
    return np.real(np.trace(rho_k @ rho_k))


def meyer_wallach(state_vector, n):
    """Meyer-Wallach entanglement Q in [0, 1] for a pure n-qubit state vector."""
    psi = np.asarray(state_vector, dtype=complex).reshape([2] * n)  # big-endian axes
    purity_sum = sum(single_qubit_purity(psi, k, n) for k in range(n))
    return 2.0 - (2.0 / n) * purity_sum


print("meyer_wallach + helpers defined")

## 2. Sanity checks: product states give Q = 0, Bell/GHZ give Q = 1

Before trusting `Q` on an ansatz, pin it to states whose entanglement we know exactly. A product
state (all `|0>`, or an independent `H` on each qubit) must give `Q = 0`. A Bell pair and a GHZ
state -- the canonical maximally-entangled states -- must give `Q = 1`.

Note each circuit ends with `.i(n - 1)`: an identity on the top qubit forces the state vector to span
all `n` qubits (length `2^n`), even when the gates only touch lower qubits. qcsim only returns
amplitudes up to the highest qubit it sees.

In [ ]:
# Product state |0000>: identity on every qubit.
zeros = Circuit()
for q in range(N_QUBITS):
    zeros.i(q)
q_zeros = meyer_wallach(state_vector_of(zeros), N_QUBITS)
print("Q(|0000>)        =", round(q_zeros, 12))

# Product state H^4: independent superposition on each qubit -> still a product state.
had = Circuit()
for q in range(N_QUBITS):
    had.h(q)
q_had = meyer_wallach(state_vector_of(had), N_QUBITS)
print("Q(H on each)     =", round(q_had, 12))

# Bell pair on 2 qubits.
bell = Circuit().h(0).cnot(0, 1)
q_bell = meyer_wallach(state_vector_of(bell), 2)
print("Q(Bell)          =", round(q_bell, 12))

# GHZ on 4 qubits.
ghz = Circuit().h(0).cnot(0, 1).cnot(1, 2).cnot(2, 3)
q_ghz = meyer_wallach(state_vector_of(ghz), N_QUBITS)
print("Q(GHZ_4)         =", round(q_ghz, 12))

# Asserts: product states are unentangled, Bell/GHZ are maximally entangled.
assert abs(q_zeros) < 1e-9, "product state |0000> must have Q == 0"
assert abs(q_had) < 1e-9, "H on each qubit is a product state -> Q == 0"
assert abs(q_bell - 1.0) < 1e-9, "Bell state must have Q == 1"
assert abs(q_ghz - 1.0) < 1e-9, "GHZ state must have Q == 1"
print("\nAll entanglement sanity checks passed.")

## 3. Three ansatze: bare rotations, hardware-efficient, strongly-entangling

We now build the architectures the GUIDE contrasts, plus a baseline:

- **Bare product-rotation circuit** -- single-qubit `RY` + `RZ` per layer and *no entangling gates*.
  This is our control: with no two-qubit gates it can only ever produce a product state, so it should
  register `Q = 0` regardless of its parameters.
- **Hardware-efficient ansatz** -- single-qubit rotations followed by a *linear chain* of
  nearest-neighbor `CNOT`s. Cheap on real hardware (only adjacent couplings), but the entanglement it
  spreads is local.
- **Strongly-entangling ansatz** -- richer `RX`-`RY`-`RZ` rotations followed by a *full ring* of
  `CNOT`s whose stride grows with the layer index, so every qubit eventually couples to every other.
  More entangling power per layer at the cost of more two-qubit gates.

Each takes a parameter tensor and ends with `.i(N_QUBITS - 1)` so the state vector always spans all
qubits.

In [ ]:
def product_rotation_circuit(params):
    """Baseline: single-qubit RY+RZ rotations, NO entangling gates.

    params shape: (N_LAYERS, N_QUBITS, 2)
    """
    c = Circuit()
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            c.ry(q, params[layer, q, 0])
            c.rz(q, params[layer, q, 1])
    c.i(N_QUBITS - 1)  # span all qubits
    return c


def hardware_efficient_circuit(params):
    """RY+RZ rotations then a linear chain of nearest-neighbor CNOTs.

    params shape: (N_LAYERS, N_QUBITS, 2)
    """
    c = Circuit()
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            c.ry(q, params[layer, q, 0])
            c.rz(q, params[layer, q, 1])
        for q in range(N_QUBITS - 1):       # nearest-neighbor entanglers
            c.cnot(q, q + 1)
    c.i(N_QUBITS - 1)
    return c


def strongly_entangling_circuit(params):
    """RX+RY+RZ rotations then a full ring of CNOTs with layer-dependent stride.

    params shape: (N_LAYERS, N_QUBITS, 3)
    """
    c = Circuit()
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            c.rx(q, params[layer, q, 0])
            c.ry(q, params[layer, q, 1])
            c.rz(q, params[layer, q, 2])
        stride = layer + 1                   # denser, longer-range coupling
        for q in range(N_QUBITS):
            c.cnot(q, (q + stride) % N_QUBITS)
    c.i(N_QUBITS - 1)
    return c


# Inspect depth / two-qubit cost on one fixed parameter draw.
np.random.seed(0)
p2_demo = np.random.uniform(0, 2 * np.pi, size=(N_LAYERS, N_QUBITS, 2))
p3_demo = np.random.uniform(0, 2 * np.pi, size=(N_LAYERS, N_QUBITS, 3))
for name, circ in [
    ("product-rotation", product_rotation_circuit(p2_demo)),
    ("hardware-efficient", hardware_efficient_circuit(p2_demo)),
    ("strongly-entangling", strongly_entangling_circuit(p3_demo)),
]:
    print(f"{name:20s} qubits={circ.qubit_count}  depth={circ.depth}")

## 4. Average entangling capability over random parameters

The entanglement an ansatz *can* reach is a property of its structure, not of one lucky parameter
setting. So we sample many random parameter vectors (seeded for reproducibility), compute `Q` for
each resulting state from its exact state vector, and average.

This average `Q` is a proxy for **entangling capability**: how much correlation the architecture
typically generates. We expect a strict ordering -- the bare rotations sit pinned at `0`, the linear
chain spreads some entanglement, and the strongly-entangling ring spreads the most.

In [ ]:
N_SAMPLES = 30  # <= 50, keeps runtime small

# Seed once, then draw every parameter tensor we need (deterministic, reproducible).
np.random.seed(0)
params_2 = np.random.uniform(0, 2 * np.pi, size=(N_SAMPLES, N_LAYERS, N_QUBITS, 2))
params_3 = np.random.uniform(0, 2 * np.pi, size=(N_SAMPLES, N_LAYERS, N_QUBITS, 3))


def average_Q(build_fn, param_samples):
    """Mean Meyer-Wallach Q over the given parameter samples (exact state vectors)."""
    qs = [meyer_wallach(state_vector_of(build_fn(p)), N_QUBITS) for p in param_samples]
    return np.array(qs)


q_product = average_Q(product_rotation_circuit, params_2)
q_hardware = average_Q(hardware_efficient_circuit, params_2)
q_strong = average_Q(strongly_entangling_circuit, params_3)

print(f"product-rotation   avg Q = {q_product.mean():.6f}   (std {q_product.std():.4f})")
print(f"hardware-efficient avg Q = {q_hardware.mean():.6f}   (std {q_hardware.std():.4f})")
print(f"strongly-entangl.  avg Q = {q_strong.mean():.6f}   (std {q_strong.std():.4f})")

# The bare rotation circuit has no two-qubit gates -> product state for ANY params -> Q == 0 exactly.
assert np.allclose(q_product, 0.0, atol=1e-9), "rotation-only circuit must be unentangled"

# Required claim: the strongly-entangling ansatz beats the bare product circuit on average.
assert q_strong.mean() > q_product.mean(), "strongly-entangling must out-entangle product circuit"

# Headline comparison: denser entanglement -> higher average Q than the linear chain.
assert q_strong.mean() > q_hardware.mean(), "strongly-entangling should out-entangle hardware-efficient"
print("\nOrdering confirmed:  strongly-entangling > hardware-efficient > product.")

## 5. Visualize the entanglement distributions

The averages tell one story; the *spread* tells another. Plot the per-sample `Q` values for each
ansatz. The product circuit collapses to a single point at `0`. The two entangling ansatze spread
toward `1`, with the strongly-entangling one shifted higher and tighter -- it reliably reaches high
entanglement, whereas the linear chain's reach depends more on the parameters it happens to draw.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0.0, 1.0, 21)
ax.hist(q_product, bins=bins, alpha=0.7, label=f"product (mean {q_product.mean():.3f})")
ax.hist(q_hardware, bins=bins, alpha=0.7, label=f"hardware-efficient (mean {q_hardware.mean():.3f})")
ax.hist(q_strong, bins=bins, alpha=0.7, label=f"strongly-entangling (mean {q_strong.mean():.3f})")
for m, c in [(q_product.mean(), "C0"), (q_hardware.mean(), "C1"), (q_strong.mean(), "C2")]:
    ax.axvline(m, color=c, linestyle="--", linewidth=1)
ax.set_xlabel("Meyer-Wallach Q")
ax.set_ylabel("count over random parameter samples")
ax.set_title(f"Entangling capability of three ansatze ({N_QUBITS} qubits, {N_SAMPLES} samples)")
ax.legend()
fig.tight_layout()
plt.show()

## 6. From entanglement to expressibility

**Expressibility** asks a related question: across random parameters, how much of state space does
an ansatz actually cover? A highly expressive ansatz spreads its output states broadly (close to a
Haar-random distribution); a weak one clusters them in a small region.

Entangling capability is one window into this. As a cheap, deterministic proxy we look at the spread
of a single observable -- `<Z_0>`, the expectation of `Z` on qubit 0 -- over our random samples. A
circuit that lands `<Z_0>` near a single value is exploring little of state space; one that spreads
`<Z_0>` across `[-1, 1]` is reaching far more states. We read `<Z_0>` straight off the state vector:
for big-endian indexing, the sign is `+1` when the top bit of the basis index is `0`, else `-1`.

In [ ]:
def expval_Z0(state_vector, n):
    """Exact <Z_0> (Z on big-endian qubit 0) from a state vector."""
    probs = np.abs(np.asarray(state_vector, dtype=complex)) ** 2
    half = 2 ** (n - 1)
    signs = np.where(np.arange(2 ** n) < half, 1.0, -1.0)  # top bit 0 -> +1, else -1
    return float(np.dot(signs, probs))


def z0_spread(build_fn, param_samples):
    return np.array([expval_Z0(state_vector_of(build_fn(p)), N_QUBITS) for p in param_samples])


z_product = z0_spread(product_rotation_circuit, params_2)
z_hardware = z0_spread(hardware_efficient_circuit, params_2)
z_strong = z0_spread(strongly_entangling_circuit, params_3)

print("<Z_0> spread (std over random params) -- wider = more of state space explored:")
print(f"  product-rotation    std = {z_product.std():.4f},  range [{z_product.min():.3f}, {z_product.max():.3f}]")
print(f"  hardware-efficient  std = {z_hardware.std():.4f},  range [{z_hardware.min():.3f}, {z_hardware.max():.3f}]")
print(f"  strongly-entangling std = {z_strong.std():.4f},  range [{z_strong.min():.3f}, {z_strong.max():.3f}]")

# <Z_0> is bounded in [-1, 1] for every sample -- a basic correctness check.
for z in (z_product, z_hardware, z_strong):
    assert np.all(z <= 1.0 + 1e-9) and np.all(z >= -1.0 - 1e-9), "<Z_0> must lie in [-1, 1]"

## 7. Now measure one circuit

Everything above used exact state vectors. On real hardware you never see the state vector -- you
get samples. To connect back to that reality, take one strongly-entangling circuit, run it on the
local simulator with finite shots, and look at the measurement histogram. The broad spread of
outcomes is the sampled shadow of the high-entanglement state we scored with `Q`.

In [ ]:
sample_circuit = strongly_entangling_circuit(params_3[0])
result = device.run(sample_circuit, shots=2000).result()
counts = result.measurement_counts

bitstrings = sorted(counts.keys())
heights = [counts[b] for b in bitstrings]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(bitstrings)), heights, color="C2")
ax.set_xticks(range(len(bitstrings)))
ax.set_xticklabels(bitstrings, rotation=90, fontsize=7)
ax.set_xlabel("measured bitstring (qubit 0 leftmost)")
ax.set_ylabel("counts (2000 shots)")
q_this = meyer_wallach(state_vector_of(sample_circuit), N_QUBITS)
ax.set_title(f"Strongly-entangling sample, Q = {q_this:.3f}: outcomes spread across many bitstrings")
fig.tight_layout()
plt.show()

print("distinct bitstrings observed:", len(bitstrings), "of", 2 ** N_QUBITS)

## Exercises

Try these to deepen the comparison. No solutions provided.

In [ ]:
# Exercise 1: Entanglement vs depth.
# Sweep N_LAYERS over, say, [1, 2, 3, 4] for the hardware-efficient and strongly-entangling
# ansatze and plot average Q vs depth. Where does each architecture saturate toward the
# Haar-random value? Does the linear chain ever catch up to the ring?
#
# TODO: wrap the builders in a function that takes n_layers as an argument, re-seed
#       np.random.seed(0) for each sweep point, and collect average_Q(...) at each depth.
#
# n_layers_grid = [1, 2, 3, 4]
# for L in n_layers_grid:
#     ...  # build, sample, average, store
# plt.plot(n_layers_grid, ...)

In [ ]:
# Exercise 2: A different entangling pattern.
# Add a third ansatz with ALL-TO-ALL entanglement: after the rotation layer, apply a CNOT
# between every pair (i, j) with i < j. Compare its average Q and <Z_0> spread against the
# linear-chain and ring ansatze. Does more entanglement always help, or does it saturate?
#
# def all_to_all_circuit(params):
#     c = Circuit()
#     for layer in range(N_LAYERS):
#         for q in range(N_QUBITS):
#             c.ry(q, params[layer, q, 0]); c.rz(q, params[layer, q, 1])
#         for i in range(N_QUBITS):
#             for j in range(i + 1, N_QUBITS):
#                 c.cnot(i, j)
#     c.i(N_QUBITS - 1)
#     return c
# TODO: sample params (seeded), compute average_Q, and compare.

## Summary

- **Meyer-Wallach Q** scores the average entanglement of a pure state via single-qubit reduced
  purities: `Q = 2 - (2/n) * sum_k Tr(rho_k^2)`. It is `0` for product states and `1` for
  Bell/GHZ states -- both verified exactly from the state vector.
- A **bare product-rotation circuit** has no two-qubit gates, so it stays at `Q = 0` for every
  parameter setting: rotations alone cannot entangle.
- The **hardware-efficient ansatz** (nearest-neighbor CNOTs) spreads moderate entanglement
  cheaply; the **strongly-entangling ansatz** (a denser CNOT ring) reaches higher average `Q`.
  Under `np.random.seed(0)` the ordering is strict: strongly-entangling > hardware-efficient >
  product.
- **Expressibility** is the broader notion of how much of state space an ansatz covers; the spread
  of `<Z_0>` over random parameters is a cheap proxy that tracks the same trend as `Q`.
- More entangling power is not free: it costs two-qubit gates (depth and noise on real hardware),
  and -- as the next notebook shows -- too much expressibility brings its own failure mode.

**Next:** [`06-barren-plateaus.ipynb`](06-barren-plateaus.ipynb) -- why highly expressive,
deeply-entangling ansatze can make gradients vanish, and how to diagnose it.